In [1]:
import pandas as pd 

df = pd.read_csv('../fixed_final_data_product.csv')

In [12]:
# keep only driver-controlled features
driver_controlled_keywords = ['THROTTLE', 'BRAKE', 'STEER', 'SPEED']
driver_cols = [c for c in df.columns if any(k in c for k in driver_controlled_keywords)]
df = df[driver_cols + ['lap_id', 'invalid_lap', 'Target_CURRENTLAPTIMEINMS']]
# Remove any null values
df = df.dropna()

# Remove laps that are too slow using IQR
Q1 = df['Target_CURRENTLAPTIMEINMS'].quantile(0.25)
Q3 = df['Target_CURRENTLAPTIMEINMS'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

df = df[(df['Target_CURRENTLAPTIMEINMS'] <= upper_bound)]


In [13]:
# -----------------------------
# LightGBM: train & report MSE
# -----------------------------
from sklearn.model_selection import train_test_split

# drop leak columns and target from predictors
X = df.drop(columns=['lap_id', 'invalid_lap', 'Target_CURRENTLAPTIMEINMS'])
# X = X.drop(columns=[c for c in X.columns if 'CURRENTLAPTIMEINMS' in c])

# set target variable
y = df['Target_CURRENTLAPTIMEINMS']

# split train and test dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [14]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score
import warnings
import lightgbm as lgb

# Suppress Python warnings
warnings.filterwarnings("ignore")

# Model training (quiet mode)
model = LGBMRegressor(
    random_state=42,
    verbosity=-1  # turns off LightGBM info/warning messages
)

model.fit(X_train, y_train)

# Make predictions
preds = model.predict(X_test)

# Evaluate performance
mse = mean_squared_error(y_test, preds)
rmse = mse ** 0.5
r2 = r2_score(y_test, preds)

print(f"MSE : {mse:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²  : {r2:,.4f}")



MSE : 327,479.39
RMSE: 572.26
R²  : 0.6839


In [15]:
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

rates = [0.1, 0.05, 0.03, 0.02, 0.01]
results = []
for lr in rates:
    m = LGBMRegressor(
        n_estimators=5000, learning_rate=lr, num_leaves=31,
        random_state=42, verbosity=-1
    )
    m.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        eval_metric="rmse",
        callbacks=[lgb.early_stopping(stopping_rounds=120)]
    )
    p = m.predict(X_test, num_iteration=m.best_iteration_)
    rmse = np.sqrt(mean_squared_error(y_test, p))
    results.append((lr, m.best_iteration_, rmse))
for lr, it, rmse in results:
    print(f"lr={lr:<5} best_iter={it:<4} RMSE={rmse:.2f} ms")


Training until validation scores don't improve for 120 rounds
Early stopping, best iteration is:
[97]	valid_0's rmse: 571.607	valid_0's l2: 326735
Training until validation scores don't improve for 120 rounds
Early stopping, best iteration is:
[261]	valid_0's rmse: 569.188	valid_0's l2: 323975
Training until validation scores don't improve for 120 rounds
Early stopping, best iteration is:
[257]	valid_0's rmse: 567.633	valid_0's l2: 322207
Training until validation scores don't improve for 120 rounds
Early stopping, best iteration is:
[529]	valid_0's rmse: 570.06	valid_0's l2: 324968
Training until validation scores don't improve for 120 rounds
Early stopping, best iteration is:
[534]	valid_0's rmse: 571.023	valid_0's l2: 326068
lr=0.1   best_iter=97   RMSE=571.61 ms
lr=0.05  best_iter=261  RMSE=569.19 ms
lr=0.03  best_iter=257  RMSE=567.63 ms
lr=0.02  best_iter=529  RMSE=570.06 ms
lr=0.01  best_iter=534  RMSE=571.02 ms


In [16]:
best_lr = 0.05  # from step 1
cands_leaves = [31, 50, 63, 95]
cands_mcs = [5, 8, 10, 15]
for nl in cands_leaves:
    for mcs in cands_mcs:
        m = LGBMRegressor(
            n_estimators=5000, learning_rate=best_lr,
            num_leaves=nl, min_child_samples=mcs,
            random_state=42, verbosity=-1
        )
        m.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              eval_metric="rmse",
              callbacks=[lgb.early_stopping(stopping_rounds=150)])
        p = m.predict(X_test, num_iteration=m.best_iteration_)
        rmse = np.sqrt(mean_squared_error(y_test, p))
        print(f"leaves={nl:<3} mcs={mcs:<2} → RMSE={rmse:.2f} ms | best_iter={m.best_iteration_}")


Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[301]	valid_0's rmse: 593.719	valid_0's l2: 352502
leaves=31  mcs=5  → RMSE=593.72 ms | best_iter=301
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[864]	valid_0's rmse: 592.213	valid_0's l2: 350716
leaves=31  mcs=8  → RMSE=592.21 ms | best_iter=864
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[2006]	valid_0's rmse: 589.15	valid_0's l2: 347097
leaves=31  mcs=10 → RMSE=589.15 ms | best_iter=2006
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[181]	valid_0's rmse: 575.057	valid_0's l2: 330690
leaves=31  mcs=15 → RMSE=575.06 ms | best_iter=181
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[133]	valid_0's rmse: 604.138	valid_0's l2: 364983
leaves=50  mcs=5  → RMSE=604.14 ms | best_iter=133
Trai

In [17]:
model = LGBMRegressor(
    n_estimators=5000,
    learning_rate=0.05,
    num_leaves=95,
    min_child_samples=15,
    subsample=0.8,          # random row sampling
    colsample_bytree=0.8,   # random feature sampling
    reg_alpha=0.1,          # L1 regularization
    reg_lambda=0.3,         # L2 regularization
    random_state=42,
    verbosity=-1
)
model.fit(X_train, y_train,
          eval_set=[(X_test, y_test)],
          eval_metric="rmse",
          callbacks=[lgb.early_stopping(stopping_rounds=150)])

preds = model.predict(X_test, num_iteration=model.best_iteration_)
mse = mean_squared_error(y_test, preds)
rmse = mse ** 0.5
print(f"RMSE: {rmse:.2f} ms | Best Iteration: {model.best_iteration_}")


Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[172]	valid_0's rmse: 571.072	valid_0's l2: 326123
RMSE: 571.07 ms | Best Iteration: 172
